# DWBI Week 2 — DW Lifecycle & Architecture
**Ali Raza** · `su92-bsdsm-f23-018` · **BSDS-7A**

Case: **PakFreight** — three transport modes, three files, three totals, zero trust.
The week ends with one trusted table: `Shipments`, 34 rows, 578,650 kg.

## Task 2 — Inmon, Kimball, and the lifecycle

### 1. Inmon vs Kimball, side by side

| Dimension | Inmon (top-down) | Kimball (bottom-up) |
|---|---|---|
| **Starting point** | The enterprise data warehouse, modelled first | One business-process data mart (here: Shipments) |
| **Modelling style** | Normalised, 3NF enterprise model (CIF) | Dimensional: facts + conformed dimensions (the Bus) |
| **Time to first value** | Slow — users wait for the enterprise model | Fast — the first mart ships in weeks |
| **Risk** | A long wait before anyone sees value; heavy upfront design | Marts drift into stovepipes if conformance is ignored |
| **Consistency enforced by** | The central normalised model; marts are derived (dependent) | Conformed dimensions shared across marts via the bus matrix |
| **Driven from** | Requirements first (SDLC: requirements → program) | Data first (CLDS: data → requirements) |

### 2. Both architectures, drawn

![Inmon CIF](charts/t2_inmon.png)

![Kimball Bus](charts/t2_kimball.png)

Data flows left to right in both: sources → staging → the integration point (Inmon's normalised EDW, Kimball's conformed dimensions) → presentation → BI. The difference is *where* integration lives — one central model versus shared dimensions.

### 3. Ten scenarios

| # | Scenario | Verdict | Why |
|---|---|---|---|
| 1 | Bank must integrate 40 systems into one enterprise "customer" for regulators | **Inmon** | One governed enterprise model must exist before any reporting |
| 2 | Startup needs a sales dashboard live in six weeks | **Kimball** | One mart, fast — value early beats enterprise purity |
| 3 | Hospital group, five systems, three-year regulatory programme | **Inmon** | Regulation + long horizon reward the governed central model |
| 4 | PakFreight builds the Shipments mart first, with conformed Date and Route | **Kimball** | Bottom-up mart, conformance via the bus |
| 5 | Retailer with a normalised core warehouse feeding departmental marts | **Inmon** | Dependent marts derived from a 3NF core is the CIF |
| 6 | Telecom moving to a lakehouse: bronze, silver, gold | **Hybrid** | Medallion layers borrow both — engineered centre, domain marts |
| 7 | Government census: the model must be agreed before loading | **Inmon** | Requirements-first, model-first is SDLC/top-down |
| 8 | E-commerce adds a Returns mart reusing the Product dimension | **Kimball** | New mart plugging into existing conformed dimensions |
| 9 | Manufacturer built six marts that no longer reconcile | **Kimball (failed)** | Bottom-up without conformance collapses back into stovepipes |
| 10 | Bank integrating a newly acquired competitor's systems | **Inmon** | Acquisition integration needs one enterprise definition first |

### 4. The Kimball lifecycle on PakFreight Iteration 1 (five phases)

- **Business requirements:** interview Ops/Finance/Sales; the one agreed question is "total weight shipped in February, one number" — and the agreed definition of a shipment (no TOTAL rows, no duplicates, kg not tons).
- **Dimensional modelling:** grain = one row per shipment; fact = `Shipments`; conformed dimensions = Date and Route (Origin/Destination); degenerate dimension = ShipmentID.
- **ETL design and development:** three staging queries (`marine_stg`, `rail_stg`, `road_stg`) bend each file into the 8-column contract — UK-locale dates, tons × 1000, `RL-` prefix, UPPER road IDs, TRIM + PROPER status, named row removals — then append into `Shipments`.
- **BI application development:** one validation page — card `SUM(WeightKG)` = 578,650, bar by `SourceSystem`, status counts 22 / 8 / 4, date-trap check on MR-1003.
- **Deployment:** publish the `.pbix` with `_stg` queries hidden (Enable load off); schedule Refresh; hand the lineage document to the auditor.

### The lifecycle as a flowchart

![Kimball lifecycle — PakFreight Iteration 1](charts/t2_lifecycle.png)

The full ten-phase Kimball lifecycle, drawn from the Mermaid source in `charts/t2_lifecycle.mmd`:
Programme Planning → Business Requirements → Technical Architecture → Product Selection →
Dimensional Modelling → Physical Design → ETL Design & Development → BI Application Development →
Deployment → Maintenance & Growth — with the feedback loop: Iteration 2 (Billing) sends new
requirements back to phase 2, reusing the conformed Date and Route dimensions.

### 5. Iteration 2 adds Billing

Build **Date** and **Route** as conformed dimensions in Iteration 1 so Billing can reuse them. "Conformed" means *the same dimension, same keys, same meaning* in every mart — a February in Billing is the identical Date row as a February in Shipments, so margin per shipment can be computed across both facts without reconciliation.

### 6. SDLC vs CLDS

SDLC runs requirements → program: you agree what to build, then build it. CLDS runs data → requirements: you look at the data first, and the requirements emerge from what you find. Warehouse projects discover requirements late because source data is always messier than anyone claimed — nobody *required* "handle the rail title rows" until someone opened `rail_shipments.xlsx` and found two of them sitting above the header. The data told us what the requirements were.

### The question to answer

"We built the Shipments mart first, with conformed Date and Route dimensions" is **bottom-up (Kimball)** — one business process first, consistency via the bus. For the other answer to be right, PakFreight would have had to build the normalised enterprise warehouse *first* and derive the Shipments mart from it afterwards.

In [ ]:
# The bus matrix is Kimball's conformance contract: which dimensions each process shares
import pandas as pd
dims = ["Date", "Customer", "Route", "Vehicle", "Driver", "Station/Port", "Commodity"]
bus = pd.DataFrame([
    ["Shipments",          "X","X","X","X","","X","X"],
    ["Billing & Invoicing","X","X","X","","","X","X"],
], columns=["Process"] + dims).set_index("Process")
print(bus.to_string())
print("\nDate is conformed first: every process happens in time, so one shared calendar")
print("is what lets Billing reuse Shipments' dimensions in Iteration 2.")